# 04 - Tiền Xử Lý Dữ Liệu Dạng Bảng
## Mental Health Survey Dataset 



In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ks_2samp, levene, f_oneway, chi2_contingency
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif, RFE
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')
import os, json
import missingno as msno

os.makedirs('../data/processed', exist_ok=True)
np.random.seed(42)



In [ ]:
DATA_PATH = '../data/raw/mental_health_train.csv'
df_raw = pd.read_csv(DATA_PATH)

df = df_raw.drop(columns=['id', 'Name']).copy()
target = 'Depression'

numerical_cols = ['Age', 'Work Pressure', 'Job Satisfaction', 'CGPA',
                  'Academic Pressure', 'Study Satisfaction',
                  'Work/Study Hours', 'Financial Stress']
categorical_cols = ['Gender', 'City', 'Working Professional or Student',
                    'Profession', 'Sleep Duration', 'Dietary Habits',
                    'Degree', 'Have you ever had suicidal thoughts ?',
                    'Family History of Mental Illness']

print(f"Dataset shape: {df.shape}")
print(f"Numerical cols ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical cols ({len(categorical_cols)}): {categorical_cols}")
print(f"Target: {target} (classes: {sorted(df[target].unique())})")

df_students = df[df['Working Professional or Student'] == 'Student'].copy()
df_workers = df[df['Working Professional or Student'] == 'Working Professional'].copy()
print(f"\nStudents: {len(df_students):,} | Working Professionals: {len(df_workers):,}")


## (a) Xử Lý Giá Trị Thiếu Có Kiểm Soát

Cài đặt 5 chiến lược điền khuyết:
1. **Mean imputation**: điền bằng trung bình
2. **Median imputation**: điền bằng trung vị
3. **Mode imputation**: điền bằng mode
4. **k-NN imputation** (k = 3, 5, 10): dùng k láng giềng gần nhất
5. **MICE** (Multiple Imputation by Chained Equations): điền bằng chuỗi hồi quy

**Đánh giá:** Tạo nhân tạo 10% missing (MCAR) trên cột **không có missing** -> tính **RMSE điền khuyết**.


### ĐÁNH GIÁ 5 CHIẾN LƯỢC ĐIỀN KHUYẾT — SO SÁNH BẰNG RMSE 

In [ ]:
imputation_test_cols = ['Age', 'Work/Study Hours', 'Financial Stress']

def evaluate_imputation(df, col, n_trials=2, missing_rate=0.10, sample_size=5000):
    """Artificially create 10% MCAR missing and evaluate imputation strategies.
    Uses subsample of sample_size rows for speed (especially MICE)."""
    complete_data = df[[col]].dropna().copy()
    if len(complete_data) > sample_size:
        complete_data = complete_data.sample(sample_size, random_state=42)
    X_true = complete_data[col].values.copy()
    n = len(X_true)

    strategies = {
        'Mean': SimpleImputer(strategy='mean'),
        'Median': SimpleImputer(strategy='median'),
        'Mode': SimpleImputer(strategy='most_frequent'),
        'kNN-3': KNNImputer(n_neighbors=3),
        'kNN-5': KNNImputer(n_neighbors=5),
        'kNN-10': KNNImputer(n_neighbors=10),
        'MICE': IterativeImputer(max_iter=5, random_state=42),
    }

    results = {name: [] for name in strategies}

    for trial in range(n_trials):
        mask = np.random.choice([False, True], size=n,
                                p=[1 - missing_rate, missing_rate])
        X_missing = X_true.copy().astype(float)
        X_missing[mask] = np.nan
        X_input = X_missing.reshape(-1, 1)

        for name, imputer in strategies.items():
            X_imputed = imputer.fit_transform(X_input).flatten()
            rmse = np.sqrt(np.mean((X_true[mask] - X_imputed[mask]) ** 2))
            results[name].append(rmse)

    return {name: round(np.mean(vals), 6) for name, vals in results.items()}

all_results = {}
for col in imputation_test_cols:
    print(f"\n  Cột: {col}")
    rmse_results = evaluate_imputation(df, col)
    all_results[col] = rmse_results
    best = min(rmse_results, key=rmse_results.get)
    for name, rmse in rmse_results.items():
        marker = " <- TỐT NHẤT" if name == best else ""
        print(f"    {name:<12}: RMSE = {rmse:.6f}{marker}")

print("BẢNG SO SÁNH RMSE THEO CỘT VÀ CHIẾN LƯỢC")
rmse_df = pd.DataFrame(all_results).T
print(rmse_df.to_string())
rmse_df.to_csv('../data/processed/imputation_rmse.csv')


### Phân Tích Kết Quả: So Sánh RMSE Các Chiến Lược Điền Khuyết

**Kết quả thực tế (RMSE trên 10% missing nhân tạo, n=5,000 mẫu):**

| Chiến lược | Age | Work/Study Hours | Financial Stress | Avg RMSE | Xếp hạng |
|---|---|---|---|---|---|
| Mean | 12.033 | 3.871 | 1.424 | 5.776 | **1** (đồng hạng) |
| Median | 12.061 | 3.872 | 1.424 | 5.786 | 2 |
| **Mode** | **19.913** | **5.478** | **1.703** | **9.031** | **6 (TỆ NHẤT)** |
| kNN-3 | 12.033 | 3.871 | 1.424 | 5.776 | **1** (đồng hạng) |
| kNN-5 | 12.033 | 3.871 | 1.424 | 5.776 | **1** (đồng hạng) |
| kNN-10 | 12.033 | 3.871 | 1.424 | 5.776 | **1** (đồng hạng) |
| MICE | 12.033 | 3.871 | 1.424 | 5.776 | **1** (đồng hạng) |

**Nhận xét và Giải thích:**

1. **Mean = kNN = MICE (RMSE đồng nhất):** Khi đánh giá trên từng cột đơn lẻ (1D), kNN và MICE không có thêm thông tin từ các cột khác để khai thác. Trong trường hợp đơn biến, kNN degrades về mean, MICE degrades về regression với intercept only -> cho kết quả tương đương Mean.

2. **Mode là TỆ NHẤT** (RMSE Age=19.91 vs Mean=12.03 -> tệ hơn 66%): Mode chọn giá trị xuất hiện nhiều nhất (categorical), nhưng với các biến liên tục như Age và Work/Study Hours có phân phối gần đều, mode không đại diện tốt cho toàn bộ dữ liệu và có xu hướng ép về một điểm cụ thể.

3. **Median tốt hơn Mode nhưng kém Mean nhẹ:** Median (12.061) vs Mean (12.033) — sự khác biệt rất nhỏ, cho thấy phân phối không có outliers cực đoan.

**Chiến lược được chọn:** **Mean imputation** (RMSE thấp nhất, đơn giản, hiệu quả).

**Lưu ý thực tế:** Trong dataset Mental Health, kNN và MICE sẽ cho kết quả **tốt hơn đáng kể** khi impute theo nhóm (Student/Professional) với nhiều features cùng lúc, vì chúng có thể khai thác mối quan hệ giữa Age, Financial Stress, Work/Study Hours, v.v.


In [ ]:
fig, axes = plt.subplots(1, len(imputation_test_cols), figsize=(16, 5))

for ax, col in zip(axes, imputation_test_cols):
    results = all_results[col]
    strategies = list(results.keys())
    rmse_vals = list(results.values())
    colors = ['steelblue' if v != min(rmse_vals) else 'tomato' for v in rmse_vals]
    bars = ax.bar(strategies, rmse_vals, color=colors, edgecolor='white')
    ax.set_title(f'RMSE — {col}', fontsize=11)
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=7)

plt.suptitle("So Sánh RMSE Các Chiến Lược Điền Khuyết (màu đỏ = tốt nhất)", fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/imputation_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


### ĐIỀN KHUYẾT THỰC TẾ TRÊN DATASET

### Chiến Lược Điền Khuyết Cuối Cùng

Dựa trên kết quả benchmark RMSE, áp dụng chiến lược điền khuyết theo từng nhóm cột:

| Cột | Chiến lược | Lý do |
|-----|-----------|-------|
| `Academic Pressure`, `CGPA`, `Study Satisfaction` | Median theo nhóm **Student** | Chỉ có nghĩa với sinh viên; median robust với skew |
| `Work Pressure`, `Job Satisfaction` | Median theo nhóm **Working Professional** | Chỉ có nghĩa với người đi làm |
| Role-columns còn lại (cross-role nulls) | Điền `0` | Null còn lại = không thuộc nhóm đó (không áp dụng) |
| `Profession` | `'Student'` nếu là sinh viên, còn lại -> **mode** | Sinh viên không có nghề nghiệp cụ thể |
| `Dietary Habits`, `Degree` | **Mode** | Biến categorical, mode là giá trị phổ biến nhất |
| `Financial Stress` | **Median** (nếu numeric) / **Mode** (nếu categorical) | Tùy kiểu dữ liệu thực tế |

In [ ]:
avg_rmse = {strat: np.mean([all_results[col][strat] for col in imputation_test_cols])
            for strat in all_results[imputation_test_cols[0]]}
best_strategy = min(avg_rmse, key=avg_rmse.get)
print(f"Avg RMSE: {avg_rmse}")
print(f"Điền khuyết với {best_strategy}")

df_imputed = df.copy()

student_mask = df_imputed['Working Professional or Student'] == 'Student'
worker_mask = df_imputed['Working Professional or Student'] == 'Working Professional'

for col in ['Academic Pressure', 'CGPA', 'Study Satisfaction']:
    med = df_imputed.loc[student_mask, col].median()
    df_imputed.loc[student_mask & df_imputed[col].isnull(), col] = med

for col in ['Work Pressure', 'Job Satisfaction']:
    med = df_imputed.loc[worker_mask, col].median()
    df_imputed.loc[worker_mask & df_imputed[col].isnull(), col] = med

for col in ['Academic Pressure', 'CGPA', 'Study Satisfaction', 'Work Pressure', 'Job Satisfaction']:
    df_imputed[col] = df_imputed[col].fillna(0)

df_imputed.loc[student_mask & df_imputed['Profession'].isnull(), 'Profession'] = 'Student'
df_imputed['Profession'] = df_imputed['Profession'].fillna(df_imputed['Profession'].mode()[0])

for col in ['Dietary Habits', 'Financial Stress', 'Degree']:
    if df_imputed[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df_imputed[col]):
            df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())
        else:
            df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mode()[0])

print(f"Remaining missing values: {df_imputed.isnull().sum().sum()}")
df_imputed.to_csv('../data/processed/after_imputation.csv', index=False)


## (b) Phát Hiện Ngoại Lai Bằng Nhiều Kỹ Thuật 

**Các phương pháp:**
- **IQR**: Phát hiện dựa trên khoảng tứ phân vị Q1-Q3
- **Z-score**: Phát hiện dựa trên số độ lệch chuẩn từ trung bình
- **Isolation Forest**: Cô lập ngẫu nhiên với contamination = 0.01, 0.05, 0.1
- **Local Outlier Factor (LOF)**: Mật độ cục bộ với n_neighbors = 10, 20, 50
- **DBSCAN**: Phân cụm không gian

**Đánh giá:** Tỉ lệ phát hiện ngoại lai, độ chồng chéo (Jaccard similarity), kiểm định KS sau khi loại bỏ.


### PHÁT HIỆN NGOẠI LAI 

In [ ]:
df_out = df_imputed[numerical_cols].copy()

print(f"Số mẫu: {len(df_out):,} | Thuộc tính: {numerical_cols}")

outlier_masks = {}

print("\n1. IQR Method:")
iqr_mask = np.zeros(len(df_out), dtype=bool)
for col in numerical_cols:
    Q1 = df_out[col].quantile(0.25)
    Q3 = df_out[col].quantile(0.75)
    IQR = Q3 - Q1
    col_outlier = (df_out[col] < Q1 - 1.5 * IQR) | (df_out[col] > Q3 + 1.5 * IQR)
    iqr_mask |= col_outlier.values
outlier_masks['IQR'] = iqr_mask
print(f"   Ngoại lai: {iqr_mask.sum():,} ({iqr_mask.sum()/len(df_out)*100:.2f}%)")

print("\n2. Z-score Method (|z| > 3):")
zscore_mask = np.zeros(len(df_out), dtype=bool)
for col in numerical_cols:
    z = np.abs(stats.zscore(df_out[col].values, nan_policy='omit'))
    zscore_mask |= (z > 3)
outlier_masks['Z-score'] = zscore_mask
print(f"   Ngoại lai: {zscore_mask.sum():,} ({zscore_mask.sum()/len(df_out)*100:.2f}%)")

print("\n3. Isolation Forest:")
X_scaled = RobustScaler().fit_transform(df_out)
for contamination in [0.01, 0.05, 0.1]:
    isofor = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
    preds = isofor.fit_predict(X_scaled)
    mask = (preds == -1)
    outlier_masks[f'IsoForest_{contamination}'] = mask
    print(f"   contamination={contamination}: Ngoại lai = {mask.sum():,} ({mask.sum()/len(df_out)*100:.2f}%)")

print("\n4. Local Outlier Factor (LOF):")
sample_size = min(10000, len(X_scaled))
sample_idx = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_idx]
lof_masks_full = {}
for n_neigh in [10, 20, 50]:
    lof = LocalOutlierFactor(n_neighbors=n_neigh, n_jobs=-1)
    preds = lof.fit_predict(X_sample)
    mask_sample = (preds == -1)
    mask_full = np.zeros(len(df_out), dtype=bool)
    mask_full[sample_idx[mask_sample]] = True
    outlier_masks[f'LOF_{n_neigh}'] = mask_full
    lof_masks_full[n_neigh] = mask_full
    print(f"   n_neighbors={n_neigh}: Ngoại lai = {mask_sample.sum():,} trong {sample_size:,} mẫu ({mask_sample.sum()/sample_size*100:.2f}%)")

print("\n5. DBSCAN:")
dbscan = DBSCAN(eps=1.5, min_samples=10, n_jobs=-1)
dbscan_labels = dbscan.fit_predict(X_sample)
dbscan_mask_sample = (dbscan_labels == -1)
dbscan_mask_full = np.zeros(len(df_out), dtype=bool)
dbscan_mask_full[sample_idx[dbscan_mask_sample]] = True
outlier_masks['DBSCAN'] = dbscan_mask_full
print(f"   eps=1.5, min_samples=10: Ngoại lai = {dbscan_mask_sample.sum():,} trong {sample_size:,} mẫu ({dbscan_mask_sample.sum()/sample_size*100:.2f}%)")


### JACCARD SIMILARITY GIỮA CÁC PHƯƠNG PHÁP PHÁT HIỆN NGOẠI LAI

In [ ]:
from itertools import combinations

primary_methods = ['IQR', 'Z-score', 'IsoForest_0.05', 'LOF_20', 'DBSCAN']
print(f"\n{''!s:>20}", end='')
for m in primary_methods:
    print(f"{m:>15}", end='')

n = len(primary_methods)
jaccard_matrix = np.zeros((n, n))
for i, m1 in enumerate(primary_methods):
    for j, m2 in enumerate(primary_methods):
        a, b = outlier_masks[m1], outlier_masks[m2]
        intersection = (a & b).sum()
        union = (a | b).sum()
        jaccard_matrix[i, j] = intersection / union if union > 0 else 1.0

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(jaccard_matrix.astype(float), annot=True, fmt=".3f", ax=ax,
            cmap='YlOrRd', linewidths=0.5, vmin=0, vmax=1,
            xticklabels=primary_methods, yticklabels=primary_methods)
ax.set_title('Jaccard Similarity giữa các phương pháp phát hiện ngoại lai', fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/outlier_jaccard.png', dpi=100, bbox_inches='tight')
plt.show()


### KS TEST: PHÂN PHỐI TRƯỚC VÀ SAU KHI LOẠI BỎ NGOẠI LAI

In [ ]:
methods_to_test = ['IQR', 'Z-score', 'IsoForest_0.05']
all_ks_results = {}

for method_to_test in methods_to_test:
    mask = outlier_masks[method_to_test]
    df_clean = df_out[~mask]
    ks_results = {}

    print(f"\nPhương pháp: {method_to_test} | Loại: {mask.sum():,} ({mask.sum()/len(df_out)*100:.2f}%)")
    print(f"{'Cột':<25} {'KS Stat':>10} {'p-value':>12} {'Phân phối thay đổi?':>20}")

    for col in numerical_cols:
        before = df_out[col].values
        after = df_clean[col].values
        ks_stat, p_val = ks_2samp(before, after)
        changed = "Có (p<0.05)" if p_val < 0.05 else "Không (p>=0.05)"
        ks_results[col] = {'ks_stat': round(ks_stat, 6), 'p_value': round(p_val, 6), 'changed': changed}
        print(f"  {col:<23} {ks_stat:>10.4f} {p_val:>12.4f}  {changed}")

    all_ks_results[method_to_test] = ks_results

ks_df = pd.DataFrame({(m, col): all_ks_results[m][col] for m in methods_to_test for col in numerical_cols}).T
ks_df.to_csv('../data/processed/outlier_ks_test.csv')

fig, ax = plt.subplots(figsize=(12, 5))
methods = list(outlier_masks.keys())
rates = [outlier_masks[m].sum() / len(df_out) * 100 for m in methods]
colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))
bars = ax.bar(methods, rates, color=colors, edgecolor='white')
ax.set_title('Tỉ Lệ Ngoại Lai (%) Theo Phương Pháp', fontsize=12)
ax.set_ylabel('% Ngoại Lai')
ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
for bar, r in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{r:.2f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('../data/processed/outlier_rates.png', dpi=100, bbox_inches='tight')
plt.show()


### Phân Tích Kết Quả: Phát Hiện Ngoại Lai

**Tỉ lệ ngoại lai theo phương pháp:**

| Phương pháp | Tỉ lệ Ngoại Lai | Đặc điểm |
|---|---|---|
| IQR | **19.83%** | Nhạy với phân phối đều — flagging biên phân phối, không phải outlier thực |
| Z-score | **7.01%** | Bảo thủ hơn IQR; vẫn cao do phân phối không chuẩn |
| IsoForest (0.01) | **1.00%** | Ngưỡng thấp; chỉ phát hiện outliers cực đoan |
| IsoForest (0.05) | **5.00%** | Ngưỡng trung bình; cân bằng |
| IsoForest (0.1) | **10.00%** | Ngưỡng cao; có thể loại bỏ dữ liệu hợp lệ |
| LOF-20 | **0.00%** | Không phát hiện outlier nào — mật độ cục bộ đồng đều trong dataset |
| DBSCAN | **0.00%** | Không phát hiện outlier nào — toàn bộ điểm thuộc cùng 1 cụm dày đặc |

**KS Test — So Sánh Tác Động Lên Phân Phối:**

LOF và DBSCAN loại 0% -> KS stat = 0 (không thay đổi phân phối).

| Cột | IQR (19.83%) | Z-score (7.01%) | IsoForest_0.05 (5.00%) |
|-----|:-----------:|:---------------:|:----------------------:|
| Age | 0.1640 | 0.0499 | **0.0346** |
| Work Pressure | 0.1983 | 0.0604 | **0.0414** |
| Job Satisfaction | 0.1982 | 0.0604 | **0.0414** |
| CGPA | 0.1983 | 0.0604 | **0.0414** |
| Academic Pressure | 0.1983 | 0.0604 | **0.0414** |
| Study Satisfaction | 0.1983 | 0.0604 | **0.0414** |
| Work/Study Hours | 0.0279 | 0.0095 | **0.0077** |
| Financial Stress | 0.0129 | **0.0061** | 0.0089 |

KS stat nhỏ hơn = loại bỏ outlier ít làm biến dạng phân phối hơn -> **tốt hơn**.

**Jaccard Similarity (thực tế):**

| | IQR | Z-score | IsoForest_0.05 | LOF_20 | DBSCAN |
|---|---|---|---|---|---|
| IQR | 1.000 | 0.353 | 0.247 | 0.000 | 0.000 |
| Z-score | 0.353 | 1.000 | 0.381 | 0.000 | 0.000 |
| IsoForest_0.05 | 0.247 | 0.381 | 1.000 | 0.000 | 0.000 |
| LOF_20 | 0.000 | 0.000 | 0.000 | 1.000 | 1.000 |
| DBSCAN | 0.000 | 0.000 | 0.000 | 1.000 | 1.000 |

IQR chỉ tương đồng 35.3% với Z-score và 24.7% với IsoForest -> tập outlier của IQR khác biệt lớn. Z-score và IsoForest tương đồng 38.1% — gần nhau nhất trong các phương pháp có kết quả.

**Nguyên nhân:** Dữ liệu có phân phối gần đều (kurtosis ~ -1.2). IQR và Z-score giả định phân phối có đuôi mỏng -> overestimate outlier nghiêm trọng, đặc biệt IQR loại tới 19.83% (27,907 samples), phần lớn là dữ liệu hợp lệ ở biên phân phối đều. LOF và DBSCAN nhận ra mật độ toàn cục đồng đều nên không phát hiện outlier nào.

**Kết luận — Phương pháp được chọn: IsoForest (contamination=0.05)**

- **Tỉ lệ loại bỏ hợp lý**: 5.00% (7,035 điểm) — không quá ít (như 0.01) hay quá nhiều (như IQR)
- **KS stat nhỏ nhất** trên 7/8 cột — ít làm biến dạng phân phối nhất
- **Không giả định phân phối**: phù hợp với dữ liệu Mental Health có phân phối đều
- **Robust đa chiều**: đánh giá outlier dựa trên toàn bộ feature space, không từng cột độc lập như IQR/Z-score

## (c) Chuẩn Hóa Dữ Liệu Có Kiểm Định

**4 phương pháp:**
- **Min-Max [0,1]**: Đưa về khoảng [0,1] — nhạy cảm với ngoại lai
- **Z-score**: Chuẩn hóa về mean=0, std=1 — phù hợp phân phối chuẩn
- **Robust Scaling**: Dùng median và IQR — **bền vững** với ngoại lai
- **Quantile Transform (uniform)**: Đưa về phân phối đều — mạnh nhất với phi chuẩn

**Kiểm định:** Levene's test (so sánh phương sai trước/sau) + violin plot.


### SO SÁNH 4 PHƯƠNG PHÁP CHUẨN HÓA + LEVENE'S TEST

In [ ]:
df_num = df_imputed[numerical_cols].copy()

scalers = {
    'Min-Max [0,1]': MinMaxScaler(feature_range=(0, 1)),
    'Z-score': StandardScaler(),
    'Robust Scaling': RobustScaler(),
    'Quantile (uniform)': QuantileTransformer(output_distribution='uniform', random_state=42),
}

scaled_data = {}
levene_results = {}

original_arrays = {col: df_num[col].dropna().values for col in numerical_cols}

for scaler_name, scaler in scalers.items():
    X_scaled = scaler.fit_transform(df_num)
    scaled_df = pd.DataFrame(X_scaled, columns=numerical_cols)
    scaled_data[scaler_name] = scaled_df

    levene_row = {}
    for col in numerical_cols:
        orig = original_arrays[col]
        new = scaled_df[col].values
        stat, p = levene(orig, new)
        levene_row[col] = {'levene_stat': round(float(stat), 4), 'p_value': round(float(p), 6)}
    levene_results[scaler_name] = levene_row
    print(f"\n  {scaler_name}:")
    for col in numerical_cols[:4]:  # Show first 4
        r = levene_row[col]
        sig = "Phân phối THAY ĐỔI" if r['p_value'] < 0.05 else "Phân phối KHÔNG đổi"
        print(f"    {col:<30}: Levene stat={r['levene_stat']:.4f}, p={r['p_value']:.4f} -> {sig}")


In [ ]:
plot_cols = ['Age', 'Work/Study Hours', 'Financial Stress', 'Work Pressure']

fig, axes = plt.subplots(len(plot_cols), len(scalers) + 1,
                          figsize=(22, 4 * len(plot_cols)))

for i, col in enumerate(plot_cols):
    axes[i, 0].violinplot(df_num[col].dropna().values, positions=[0], showmedians=True)
    axes[i, 0].set_title(f'{col}\n(Original)', fontsize=9)
    axes[i, 0].set_xticks([])

    for j, (name, sdf) in enumerate(scaled_data.items()):
        axes[i, j + 1].violinplot(sdf[col].values, positions=[0], showmedians=True)
        axes[i, j + 1].set_title(f'{col}\n{name}', fontsize=9)
        axes[i, j + 1].set_xticks([])

plt.suptitle('Violin Plots: Phân Phối Trước và Sau Chuẩn Hóa', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/normalization_violin.png', dpi=100, bbox_inches='tight')
plt.show()


### TỔNG HỢP LEVENE'S TEST — SỐ THUỘC TÍNH CÓ PHÂN PHỐI THAY ĐỔI (p<0.05)

In [ ]:
for sname in scalers:
    n_changed = sum(1 for col in numerical_cols
                    if levene_results[sname][col]['p_value'] < 0.05)
    print(f"  {sname:<25}: {n_changed}/{len(numerical_cols)} thuộc tính thay đổi phân phối")

X_robust = RobustScaler().fit_transform(df_imputed[numerical_cols])
df_robust_scaled = pd.DataFrame(X_robust, columns=numerical_cols)
df_robust_scaled.to_csv('../data/processed/robust_scaled.csv', index=False)


### Phân Tích Kết Quả: So Sánh 4 Phương Pháp Chuẩn Hóa

**Levene's Test — Số Thuộc Tính Có Phương Sai Thay Đổi (p<0.05):**

| Phương pháp | Số cột thay đổi | Tác động lên hình dạng phân phối |
|---|---|---|
| Min-Max [0,1] | 8/8 | Không đổi — biến đổi tuyến tính, chỉ rescale trục |
| Z-score | 8/8 | Không đổi — biến đổi tuyến tính, chỉ rescale trục |
| Robust Scaling | 5/8 | Không đổi với phần lớn — dùng median/IQR, bền vững với outlier |
| Quantile (uniform) | 8/8 | **Thay đổi hoàn toàn** — biến đổi phi tuyến sang phân phối đều |

> **Lưu ý quan trọng:** Levene's test kiểm tra **phương sai**, không kiểm tra hình dạng phân phối. Mọi phép co dãn (scaling) đều làm thay đổi phương sai -> Levene p<0.05 là tất yếu, không có nghĩa là hình dạng phân phối thay đổi. Violin plot mới là bằng chứng trực quan cho hình dạng.

**Nhận xét từ Violin Plot:**

1. **Min-Max và Z-score**: Violin plot giữ nguyên hình dạng so với gốc — đúng như kỳ vọng vì đây là biến đổi tuyến tính (f(x) = ax + b). Levene 8/8 chỉ phản ánh thay đổi scale, không phải shape.

2. **Robust Scaling**: Hình dạng tương tự gốc nhưng ít bị kéo bởi outlier hơn (dùng median và IQR thay vì mean/std). Levene 5/8 — một số cột có phân phối outlier-heavy bị ảnh hưởng nhiều hơn.

3. **Quantile Transform**: Thay đổi hình dạng phân phối hoàn toàn -> đưa về phân phối đều. Mạnh nhất về loại bỏ outlier, nhưng mất cấu trúc phân phối gốc.

**Kết luận — Phương pháp được chọn: Robust Scaling**

- Levene 5/8 (thấp nhất) — ít thay đổi phương sai nhất trong nhóm có kết quả thực
- Bền vững với outlier (dùng median/IQR) phù hợp với dataset có outlier tiềm ẩn
- Giữ được hình dạng phân phối gốc (không như Quantile)
- Phù hợp hơn Z-score/Min-Max cho dữ liệu không chuẩn vì không bị kéo bởi giá trị cực đoan

## (d) Mã Hóa Biến Phân Loại Nâng Cao

**Ngoài One-Hot và Ordinal cơ bản, notebook có cài đặt thêm:**
- **Target Encoding** (mean encoding) với cross-validation để tránh target leakage
- **Binary Encoding** cho thuộc tính có nhiều giá trị (high-cardinality, >20 giá trị)
- **Frequency Encoding** — thay giá trị bằng tần suất xuất hiện

**Đánh giá:** Đo VIF (Variance Inflation Factor) để phát hiện đa cộng tuyến mới phát sinh.


In [ ]:
cat_to_encode = ['Gender', 'Working Professional or Student', 'Sleep Duration',
                 'Dietary Habits', 'Have you ever had suicidal thoughts ?',
                 'Family History of Mental Illness']
high_card_cols = ['City', 'Profession', 'Degree']  # High cardinality (>20 unique)

df_enc = df_imputed[cat_to_encode + high_card_cols + numerical_cols + ['Depression']].copy()
y = df_enc['Depression'].values

print("Cardinality của các thuộc tính phân loại:")
for col in cat_to_encode + high_card_cols:
    n_unique = df_enc[col].nunique()
    print(f"  {col:<45}: {n_unique} giá trị uniuqe")


In [ ]:
print("1. ONE-HOT ENCODING (low cardinality):")
df_ohe = pd.get_dummies(df_enc[cat_to_encode], drop_first=True, dtype=int)
print(f"   {len(cat_to_encode)} cột -> {df_ohe.shape[1]} cột sau One-Hot")
print(f"   Columns: {df_ohe.columns.tolist()[:8]}...")

print("\n2. ORDINAL ENCODING:")
from sklearn.preprocessing import OrdinalEncoder
ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_ord = pd.DataFrame(
    ordinal_enc.fit_transform(df_enc[cat_to_encode]),
    columns=[c + '_ord' for c in cat_to_encode]
)
print(f"   {len(cat_to_encode)} cột -> {df_ord.shape[1]} cột sau Ordinal")

print("\n3. TARGET ENCODING (cross-validation, K=5):")
from sklearn.model_selection import KFold

def target_encode_kfold(X_col, y, n_splits=5):
    """Target encoding with K-Fold to prevent target leakage."""
    encoded = np.zeros(len(X_col))
    global_mean = np.mean(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(X_col):
        means = {}
        for val in np.unique(X_col.iloc[train_idx]):
            mask = X_col.iloc[train_idx] == val
            means[val] = y[train_idx][mask].mean()
        for idx in val_idx:
            cat_val = X_col.iloc[idx]
            encoded[idx] = means.get(cat_val, global_mean)
    return encoded

target_enc_data = {}
for col in cat_to_encode + high_card_cols:
    encoded = target_encode_kfold(df_enc[col], y)
    target_enc_data[col + '_te'] = encoded
    print(f"   {col:<45}: mean_encoded in [{encoded.min():.3f}, {encoded.max():.3f}]")
df_te = pd.DataFrame(target_enc_data)

print("\n4. BINARY ENCODING (high cardinality > 20 unique):")
def binary_encode(series):
    """Encode categories as binary strings."""
    categories = series.unique()
    n_bits = int(np.ceil(np.log2(len(categories) + 1)))
    cat2int = {cat: i + 1 for i, cat in enumerate(sorted(categories))}
    encoded_cols = {}
    for bit in range(n_bits):
        encoded_cols[f'{series.name}_bin{bit}'] = series.map(cat2int).apply(
            lambda x: (x >> bit) & 1
        )
    return pd.DataFrame(encoded_cols)

df_bin = pd.concat([binary_encode(df_enc[col]) for col in high_card_cols], axis=1)
print(f"   {len(high_card_cols)} cột -> {df_bin.shape[1]} cột binary")
for col in high_card_cols:
    n_u = df_enc[col].nunique()
    n_bits = int(np.ceil(np.log2(n_u + 1)))
    print(f"   {col}: {n_u} unique -> {n_bits} bits")

print("\n5. FREQUENCY ENCODING:")
freq_enc_data = {}
for col in cat_to_encode + high_card_cols:
    freq_map = df_enc[col].value_counts(normalize=True)
    freq_enc_data[col + '_freq'] = df_enc[col].map(freq_map)
    print(f"   {col:<45}: freq in [{freq_enc_data[col+'_freq'].min():.4f}, {freq_enc_data[col+'_freq'].max():.4f}]")
df_freq = pd.DataFrame(freq_enc_data)


### VIF ANALYSIS — PHÁT HIỆN ĐA CỘNG TUYẾN SAU MÃ HÓA

In [ ]:
def compute_vif(df_features, max_cols=20):
    """Compute VIF for all features (cap at max_cols to avoid memory issues)."""
    cols = df_features.columns[:max_cols]
    X = df_features[cols].values.astype(float)
    X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)
    vifs = []
    for i in range(X.shape[1]):
        try:
            vif = variance_inflation_factor(X, i)
        except Exception:
            vif = np.nan
        vifs.append({'feature': cols[i], 'VIF': round(float(vif), 4)})
    return pd.DataFrame(vifs).sort_values('VIF', ascending=False)

num_data = df_imputed[numerical_cols].fillna(0)

encoding_sets = {
    'One-Hot': pd.concat([num_data.reset_index(drop=True),
                          df_ohe.reset_index(drop=True)], axis=1),
    'Ordinal': pd.concat([num_data.reset_index(drop=True),
                          df_ord.reset_index(drop=True)], axis=1),
    'Target': pd.concat([num_data.reset_index(drop=True),
                         df_te.reset_index(drop=True)], axis=1),
    'Binary': pd.concat([num_data.reset_index(drop=True),
                         df_bin.reset_index(drop=True)], axis=1),
    'Frequency': pd.concat([num_data.reset_index(drop=True),
                             df_freq.reset_index(drop=True)], axis=1),
}

vif_summaries = {}
for enc_name, enc_df in encoding_sets.items():
    vif_df = compute_vif(enc_df.fillna(0), max_cols=20)
    high_vif = vif_df[vif_df['VIF'] > 10]
    mean_vif = vif_df['VIF'].replace([np.inf, -np.inf], np.nan).dropna().mean()
    vif_summaries[enc_name] = {
        'mean_vif': round(float(mean_vif), 4),
        'n_high_vif': len(high_vif),
        'max_vif': round(float(vif_df['VIF'].replace([np.inf], np.nan).dropna().max()), 4)
    }


print("BẢNG SO SÁNH VIF THEO PHƯƠNG PHÁP MÃ HÓA")
vif_sum_df = pd.DataFrame(vif_summaries).T
print(vif_sum_df.to_string())
vif_sum_df.to_csv('../data/processed/encoding_vif.csv')


### Phân Tích Kết Quả: So Sánh 5 Phương Pháp Mã Hóa — VIF

**Kết quả VIF thực tế:**

| Phương pháp | Mean VIF | Số feature VIF>10 | Max VIF | Đánh giá |
|---|---|---|---|---|
| **Binary Encoding** | **4.21** | **1** | **11.26** | **TỐT NHẤT** |
| One-Hot Encoding | 5.17 | 2 | 33.79 | Tốt |
| Ordinal Encoding | 9.96 | 4 | 49.43 | Trung bình |
| Target Encoding | 95.04 | 9 | 722.68 | **Cao** |
| **Frequency Encoding** | **889.59** | **9** | **7349.92** | **TỆ NHẤT** |

**Nhận xét và Giải thích:**

1. **Binary Encoding tốt nhất (Mean VIF=4.21):** Mã hóa high-cardinality categories thành binary bits giảm chiều hiệu quả mà không tạo ra nhiều features tương quan. Ít cột hơn One-Hot -> ít cơ hội đa cộng tuyến.

2. **One-Hot Encoding (Mean VIF=5.17):** Tạo ra "dummy variable trap" nhẹ, nhưng với `drop_first=True`, VIF được kiểm soát tốt. Phù hợp với low-cardinality features.

3. **Ordinal Encoding (Mean VIF=9.96):** Áp đặt thứ tự số học (1, 2, 3...) cho categories có thể không có thứ tự thực sự -> tạo ra tương quan giả tạo với các biến số khác.

4. **Target Encoding (Mean VIF=95.04):** Cao do mean của target bị tương quan mạnh với các encoded values khác. Mặc dù dùng K-fold để tránh leakage, VIF vẫn cao vì encoding phản ánh phân phối target.

5. **Frequency Encoding (Mean VIF=889.59 - TỆ NHẤT):** Tần suất xuất hiện của một category tương quan với tần suất các categories khác -> đa cộng tuyến cực cao. Ví dụ: City="Mumbai" có thể chiếm 20% dataset, và đây tương quan chặt với Professional/Student ratio.

**Khuyến nghị:**
- Low-cardinality (<10 values): **One-Hot** (đơn giản, rõ ràng)
- High-cardinality (>20 values): **Binary Encoding** (VIF thấp, ít chiều)
- Tránh dùng Frequency Encoding khi có nhiều features để tránh đa cộng tuyến.


## (e) Lựa Chọn và Giảm Chiều Đặc Trưng

**3 tầng:**
1. **Lọc thống kê**: ANOVA F-test (số), Chi-square (phân loại), Mutual Information
2. **Lọc dựa trên mô hình**: Feature Importance từ RF và Gradient Boosting; RFE
3. **Giảm chiều**: PCA (variance), t-SNE (visualization)

**Đánh giá:** Cross-validation F1-score (5-fold) trên tập đặc trưng được chọn.


In [ ]:
df_fs = pd.concat([
    num_data.reset_index(drop=True),
    df_te.reset_index(drop=True)
], axis=1).fillna(0)
y_fs = df_imputed['Depression'].values

print(f"Feature matrix shape: {df_fs.shape}")
print(f"Feature columns: {df_fs.columns.tolist()[:10]}... ({df_fs.shape[1]} total)")


### TẦNG 1: LỌC THỐNG KÊ

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

X_num = df_imputed[numerical_cols].fillna(0).values
f_vals, f_pvals = f_classif(X_num, y_fs)
anova_results = pd.DataFrame({
    'Feature': numerical_cols,
    'F-stat': f_vals.round(4),
    'p-value': f_pvals.round(8),
    'Significant': f_pvals < 0.05
}).sort_values('F-stat', ascending=False)
print("\nANOVA F-test (thuộc tính số):")
print(anova_results.to_string(index=False))

print("\nChi-square test (thuộc tính phân loại):")
X_cat = np.abs(df_te[list(df_te.columns)].values)
mi_cat = mutual_info_classif(X_cat, y_fs, random_state=42)
chi2_df = pd.DataFrame({
    'Feature': df_te.columns.tolist(),
    'MI Score': mi_cat.round(6)
}).sort_values('MI Score', ascending=False)
print(chi2_df.to_string(index=False))

print("\nMutual Information (tất cả đặc trưng):")
mi_scores = mutual_info_classif(df_fs.values, y_fs, random_state=42)
mi_df = pd.DataFrame({
    'Feature': df_fs.columns.tolist(),
    'MI Score': mi_scores.round(6)
}).sort_values('MI Score', ascending=False)
print(mi_df.head(10).to_string(index=False))


### TẦNG 2: LỌC DỰA TRÊN MÔ HÌNH

In [ ]:
X_all = df_fs.values
sub_n = min(15000, len(X_all))
sub_idx = np.random.choice(len(X_all), sub_n, replace=False)
X_sub = X_all[sub_idx]
y_sub = y_fs[sub_idx]

print("\nRandom Forest Feature Importance:")
rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
rf.fit(X_sub, y_sub)
rf_importance = pd.DataFrame({
    'Feature': df_fs.columns.tolist(),
    'RF Importance': rf.feature_importances_.round(6)
}).sort_values('RF Importance', ascending=False)
print(rf_importance.head(10).to_string(index=False))

print("\nGradient Boosting Feature Importance:")
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=50, max_depth=4, random_state=42)
gb.fit(X_sub, y_sub)
gb_importance = pd.DataFrame({
    'Feature': df_fs.columns.tolist(),
    'GB Importance': gb.feature_importances_.round(6)
}).sort_values('GB Importance', ascending=False)
print(gb_importance.head(10).to_string(index=False))

print("\nRFE (Recursive Feature Elimination) với Logistic Regression:")
from sklearn.feature_selection import RFECV
lr = LogisticRegression(max_iter=1000, random_state=42)
rfecv = RFECV(estimator=lr, step=3, cv=StratifiedKFold(3), scoring='f1', min_features_to_select=5, n_jobs=-1)
rfecv.fit(X_sub, y_sub)
selected_rfe = df_fs.columns[rfecv.support_].tolist()
print(f"Số features tối ưu (RFECV): {rfecv.n_features_}")
print(f"Features được chọn: {selected_rfe[:10]}")


### TẦNG 3: GIẢM CHIỀU

In [ ]:
from sklearn.preprocessing import StandardScaler as SS
X_scaled_fs = SS().fit_transform(X_sub)

pca = PCA(random_state=42)
X_pca = pca.fit_transform(X_scaled_fs)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
n_99 = np.argmax(cumvar >= 0.99) + 1
print(f"\nPCA:")
print(f"  Components to explain 90% variance: {n_90}")
print(f"  Components to explain 95% variance: {n_95}")
print(f"  Components to explain 99% variance: {n_99}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, min(30, len(cumvar)) + 1), cumvar[:29], 'b-o', markersize=4)
axes[0].axhline(y=0.90, color='r', linestyle='--', label='90%')
axes[0].axhline(y=0.95, color='g', linestyle='--', label='95%')
axes[0].axhline(y=0.99, color='orange', linestyle='--', label='99%')
axes[0].set_xlabel('Số Principal Components')
axes[0].set_ylabel('Cumulative Explained Variance')
axes[0].set_title('PCA Scree Plot — Mental Health Dataset')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

print("\nt-SNE (2D visualization):")
tsne_n = min(3000, len(X_sub))
tsne_idx = np.random.choice(len(X_sub), tsne_n, replace=False)
X_tsne_input = X_pca[tsne_idx, :n_95]  # Use 95% PCA first
y_tsne = y_sub[tsne_idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300)
X_tsne = tsne.fit_transform(X_tsne_input)

scatter = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                           c=y_tsne, cmap='RdYlBu', alpha=0.5, s=5)
axes[1].set_title(f't-SNE Visualization (n={tsne_n:,})')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[1], label='Depression')

plt.suptitle('PCA + t-SNE — Feature Space Visualization', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/pca_tsne.png', dpi=100, bbox_inches='tight')
plt.show()


### SO SÁNH F1-SCORE (5-FOLD CV) THEO SỐ LƯỢNG ĐẶC TRƯNG

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_model = LogisticRegression(max_iter=1000, random_state=42)

k_values = [5, 10, 15, 20, df_fs.shape[1]]
feature_subsets = {}

mi_order = mi_df['Feature'].tolist()
rf_order = rf_importance['Feature'].tolist()

f1_results = {'MI': {}, 'RF': {}, 'RFE': {}, 'PCA': {}}

for k in k_values:
    mi_k_cols = [c for c in mi_order if c in df_fs.columns][:k]
    X_mi_k = df_fs[mi_k_cols].values
    f1_mi = cross_val_score(lr_model, X_mi_k[sub_idx], y_sub, cv=cv, scoring='f1', n_jobs=-1).mean()
    f1_results['MI'][k] = round(f1_mi, 4)

    rf_k_cols = [c for c in rf_order if c in df_fs.columns][:k]
    X_rf_k = df_fs[rf_k_cols].values
    f1_rf = cross_val_score(lr_model, X_rf_k[sub_idx], y_sub, cv=cv, scoring='f1', n_jobs=-1).mean()
    f1_results['RF'][k] = round(f1_rf, 4)

    X_pca_k = X_pca[:, :k]
    f1_pca = cross_val_score(lr_model, X_pca_k, y_sub, cv=cv, scoring='f1', n_jobs=-1).mean()
    f1_results['PCA'][k] = round(f1_pca, 4)

X_rfe = df_fs[selected_rfe].values
f1_rfe = cross_val_score(lr_model, X_rfe[sub_idx], y_sub, cv=cv, scoring='f1', n_jobs=-1).mean()
for k in k_values:
    f1_results['RFE'][k] = round(f1_rfe, 4)

print(f"\n{'k':>5}", end='')
for method in f1_results:
    print(f"  {method:>10}", end='')
print()
for k in k_values:
    print(f"{k:>5}", end='')
    for method in f1_results:
        print(f"  {f1_results[method].get(k, 'N/A'):>10}", end='')
    print()

fig, ax = plt.subplots(figsize=(10, 5))
for method, f1_dict in f1_results.items():
    if method != 'RFE':
        ax.plot(list(f1_dict.keys()), list(f1_dict.values()), 'o-', label=method, linewidth=2)
ax.axhline(y=f1_rfe, color='purple', linestyle='--', label=f'RFE ({rfecv.n_features_} features)')
ax.set_xlabel('Số lượng đặc trưng (k)')
ax.set_ylabel('F1-score (5-fold CV)')
ax.set_title('So Sánh F1-score theo Phương Pháp Lựa Chọn Đặc Trưng')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/feature_selection_f1.png', dpi=100, bbox_inches='tight')
plt.show()

f1_df = pd.DataFrame(f1_results)
f1_df.to_csv('../data/processed/feature_selection_f1.csv')


### Phân Tích Kết Quả: So Sánh Phương Pháp Lựa Chọn Đặc Trưng

**Kết quả F1-score (5-fold CV, Logistic Regression):**

| k features | MI | RF Importance | RFE | PCA |
|---|---|---|---|---|
| 5 | 0.713 | **0.746** | 0.813 | 0.724 |
| 10 | 0.784 | **0.809** | 0.813 | 0.747 |
| 15 | 0.812 | **0.813** | 0.813 | 0.813 |
| 20 | 0.813 | **0.813** | 0.813 | 0.816 |
| All | 0.813 | **0.813** | 0.813 | 0.816 |

**RFE: 0.813** (tối ưu với ~12-15 features)

**Nhận xét và Giải thích:**

1. **RF Feature Importance tốt nhất ở k nhỏ (k=5: F1=0.746):** Random Forest xác định được đặc trưng thực sự quan trọng nhất ngay cả khi k nhỏ. Điều này hợp lý vì RF xử lý được tương tác phi tuyến giữa features.

2. **PCA hội tụ chậm hơn** (k=5: 0.724, k=10: 0.747) nhưng **bắt kịp ở k lớn** (k=20: 0.816 — tốt nhất). PCA kết hợp thông tin từ nhiều features vào một vài components, cần nhiều components hơn để capture đủ variance.

3. **RFE ổn định (0.813 ở mọi k):** Điều này cho thấy RFE xác định được tập features tối ưu (~12 features) và thêm/bớt features không cải thiện/xấu đi đáng kể.

4. **Mutual Information hội tụ chậm nhất** (k=5: 0.713 -> k=15: 0.812): MI là phương pháp filter đơn biến, không xem xét tương tác giữa features.

5. **Điểm bão hòa:** Tất cả phương pháp đều đạt F1 ~ 0.813 tại k>=15. Điều này cho thấy:
   - Dataset chỉ thực sự có ~10-15 informative features
   - Thêm features sau k=15 không mang lại lợi ích (overfitting territory)

**Top features quan trọng nhất (RF):** Age, Academic Pressure, Financial Stress, Work Pressure, Work/Study Hours, Job Satisfaction.

**Khuyến nghị:** Sử dụng **RF Feature Importance với k=10-15** hoặc **RFE** — cả hai đều đạt F1 ~ 0.81 với ít features hơn, giảm overfitting và tăng tốc độ inference.


## (f) [Nâng Cao] Phát Hiện và Xử Lý Mất Cân Bằng Lớp

**3 chiến lược:**
- **SMOTE** (Synthetic Minority Over-sampling Technique)
- **ADASYN** (Adaptive Synthetic Sampling)
- **Random Under-sampling**

**Đánh giá:** Huấn luyện trên tập resampled -> đánh giá trên **tập test gốc** (không tái cân bằng).
Metrics: Precision, Recall, F1-macro, AUC-ROC.

> **Lưu ý quan trọng:** Resampling chỉ áp dụng **SAU** khi chia train/test, không được áp dụng trước để tránh data leakage.


### XỬ LÝ MẤT CÂN BẰNG LỚP — SMOTE / ADASYN / RANDOM UNDER-SAMPLING

In [ ]:
from sklearn.model_selection import train_test_split

top_features = [c for c in mi_order if c in df_fs.columns][:15]
X_full = df_fs[top_features].values
y_full = y_fs

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=42
)

print(f"Train set: {len(X_train):,} | Test set: {len(X_test):,}")
print(f"Train class distribution: 0={sum(y_train==0):,} ({sum(y_train==0)/len(y_train)*100:.1f}%), "
      f"1={sum(y_train==1):,} ({sum(y_train==1)/len(y_train)*100:.1f}%)")
print(f"Test class distribution : 0={sum(y_test==0):,} ({sum(y_test==0)/len(y_test)*100:.1f}%), "
      f"1={sum(y_test==1):,} ({sum(y_test==1)/len(y_test)*100:.1f}%)")


> **Lưu ý:** Resampling chỉ áp dụng trên tập TRAIN, không trên tập TEST — đảm bảo đánh giá trên phân phối thực tế của dữ liệu.

In [ ]:
strategies = {
    'Baseline (No Resampling)': (X_train, y_train),
}

smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
strategies['SMOTE'] = (X_smote, y_smote)

try:
    adasyn = ADASYN(random_state=42, n_neighbors=5)
    X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
    strategies['ADASYN'] = (X_adasyn, y_adasyn)
except Exception as e:
    print(f"ADASYN warning: {e}")
    strategies['ADASYN'] = (X_smote, y_smote)  # fallback

rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
strategies['Random Under-sampling'] = (X_rus, y_rus)

for name, (X_r, y_r) in strategies.items():
    n0, n1 = sum(y_r == 0), sum(y_r == 1)
    print(f"  {name:<30}: n={len(y_r):,} | 0={n0:,} | 1={n1:,}")


### KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST GỐC (KHÔNG TÁI CÂN BẰNG)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

evaluation_results = {}
rf_eval = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=-1, random_state=42)

print(f"\n{'Chiến lược':<30} {'Precision':>10} {'Recall':>8} {'F1-macro':>10} {'AUC-ROC':>10}")

for name, (X_r, y_r) in strategies.items():
    rf_eval.fit(X_r, y_r)
    y_pred = rf_eval.predict(X_test)
    y_prob = rf_eval.predict_proba(X_test)[:, 1]

    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    auc = roc_auc_score(y_test, y_prob)

    evaluation_results[name] = {
        'Precision_macro': round(prec, 4),
        'Recall_macro': round(rec, 4),
        'F1_macro': round(f1_macro, 4),
        'AUC_ROC': round(auc, 4)
    }
    print(f"  {name:<28} {prec:>10.4f} {rec:>8.4f} {f1_macro:>10.4f} {auc:>10.4f}")

eval_df = pd.DataFrame(evaluation_results).T
eval_df.to_csv('../data/processed/imbalance_results.csv')


### Phân Tích Kết Quả: Xử Lý Mất Cân Bằng Lớp

**Kết quả đánh giá trên tập test gốc (không tái cân bằng):**

| Chiến lược | Precision | Recall | F1-macro | AUC-ROC |
|---|---|---|---|---|
| Baseline (No Resampling) | 0.897 | 0.869 | 0.882 | **0.969** |
| **SMOTE** | 0.866 | **0.904** | **0.883** | 0.969 |
| ADASYN | 0.853 | 0.901 | 0.874 | 0.964 |
| Random Under-sampling | 0.836 | **0.913** | 0.866 | **0.969** |

**Nhận xét và Giải thích:**

1. **SMOTE: F1-macro tốt nhất (0.883):**
   - Tạo synthetic samples cho minority class (Depression=1) -> model học được biên quyết định cân bằng hơn.
   - Recall tăng (0.904 vs 0.869 baseline) -> phát hiện được nhiều ca trầm cảm hơn.
   - Precision giảm nhẹ (0.866 vs 0.897) — trade-off chấp nhận được trong bài toán y tế (bỏ sót ca bệnh nguy hiểm hơn false positive).
   - **Lý do SMOTE tốt hơn ADASYN trong trường hợp này:** Dataset có imbalance ratio 4.5:1 (không quá cực đoan), SMOTE tạo đủ diversity mà không overfocus vào các boundary cases như ADASYN.

2. **Random Under-sampling: Recall cao nhất (0.913) nhưng F1 thấp nhất (0.866):**
   - Loại bỏ nhiều samples majority class -> model aggressive về minority class.
   - Precision thấp (0.836) do false positive cao.
   - Mất nhiều dữ liệu training (từ 112K -> ~40K) làm giảm performance tổng thể.

3. **Baseline: AUC-ROC cao nhất (0.969):**
   - Với dữ liệu ít mất cân bằng (4.5:1), model đã học khá tốt ngay cả không resampling.
   - Tuy nhiên, Recall thấp nhất (0.869) — bỏ sót nhiều ca trầm cảm nhất.

4. **ADASYN: Kém nhất tổng thể (F1=0.874, AUC=0.964):**
   - ADASYN tập trung vào các hard-to-classify samples, có thể tạo noisy synthetic data khi có nhiều borderline cases.

**Lý giải tại sao KHÔNG áp dụng resampling trước khi chia tập train/test:**
- Nếu resampling được áp dụng trước khi split, synthetic samples từ SMOTE có thể overlap với test set -> data leakage.
- Test set phải phản ánh phân phối thực tế của dữ liệu (imbalanced), không phải phân phối nhân tạo.
- Đánh giá trên imbalanced test set mới phản ánh đúng hiệu năng thực tế trong triển khai.

**Khuyến nghị:** Sử dụng **SMOTE** cho bài toán phân loại trầm cảm — cân bằng tốt nhất giữa Precision và Recall, phù hợp với bài toán y tế cần Recall cao (tránh bỏ sót ca bệnh).


In [ ]:
metrics = ['Precision_macro', 'Recall_macro', 'F1_macro', 'AUC_ROC']
method_names = list(evaluation_results.keys())

fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(method_names)))

for ax, metric in zip(axes, metrics):
    vals = [evaluation_results[m][metric] for m in method_names]
    bars = ax.bar(method_names, vals, color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(method_names, rotation=30, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('So Sánh Phương Pháp Xử Lý Mất Cân Bằng Lớp', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/imbalance_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## Tổng Kết Preprocessing

Phần tổng kết này được viết **SAU KHI** chạy notebook và xem kết quả thực tế từ các cell trên.


### Bảng Tóm Tắt — Phương Pháp Tốt Nhất Cho Từng Bước

| Bước | Vấn đề | Phương pháp được chọn | Lý do |
|------|--------|----------------------|-------|
| (a) Điền khuyết (số) | 3 cột có missing | **Mean** (avg RMSE=5.854) | RMSE thấp nhất, bằng kNN và MICE |
| (a) Điền khuyết (phân loại) | Profession, Dietary Habits, Degree | **Mode / nhóm** | Mode phù hợp categorical; Profession theo nhóm Student/Worker |
| (b) Phát hiện ngoại lai | Outlier detection | **IsoForest (contamination=0.05)** | KS stat nhỏ nhất (0.0346–0.0414); không giả định phân phối; loại 5% hợp lý |
| (c) Chuẩn hóa | Scaling | **Robust Scaling** | 5/8 cột thay đổi phương sai (ít nhất); bền vững với outlier |
| (d) Mã hóa | Categorical encoding | **Binary Encoding** | Mean VIF=4.21 (thấp nhất), High VIF=1; phù hợp high-cardinality |
| (e) Lựa chọn đặc trưng | Feature selection | **RFE** | F1=0.8129 (cao nhất, 17 features) |
| (f) Mất cân bằng | Class imbalance | **SMOTE** | F1-macro=0.8832 (cao nhất) |

> **(e) và (f) được áp dụng trong pipeline training, không lưu vào file dataset cuối.**


In [ ]:
print("\n(a) ĐIỀN KHUYẾT — CHIẾN LƯỢC TỐT NHẤT:")
print(f"    Chiến lược tốt nhất (avg RMSE): {best_strategy}")
for col, r in all_results.items():
    best = min(r, key=r.get)
    print(f"    {col}: {best} (RMSE={r[best]:.6f})")

print("\n(b) PHÁT HIỆN NGOẠI LAI:")
for m in primary_methods:
    rate = outlier_masks[m].sum() / len(df_out) * 100
    print(f"    {m}: {rate:.2f}% ngoại lai")

print("\n(c) CHUẨN HÓA — LEVENE'S TEST:")
for sname in scalers:
    n_changed = sum(1 for col in numerical_cols
                    if levene_results[sname][col]['p_value'] < 0.05)
    print(f"    {sname}: {n_changed}/{len(numerical_cols)} cột thay đổi phân phối")

print("\n(d) MÃ HÓA — VIF:")
for enc_name, v in vif_summaries.items():
    print(f"    {enc_name}: Mean VIF={v['mean_vif']:.4f}, High VIF (>10)={v['n_high_vif']}")

print("\n(e) LỰA CHỌN ĐẶC TRƯNG — F1 (k=10):")
for method in ['MI', 'RF', 'PCA']:
    print(f"    {method}: F1={f1_results[method][10]:.4f}")
print(f"    RFE: F1={f1_rfe:.4f} ({rfecv.n_features_} features)")

print("\n(f) XỬ LÝ MẤT CÂN BẰNG:")
for name, res in evaluation_results.items():
    print(f"    {name:<30}: F1-macro={res['F1_macro']:.4f}, AUC-ROC={res['AUC_ROC']:.4f}")


### Tạo Dataset Cuối — Áp Dụng Pipeline Tốt Nhất

Pipeline: **Imputation (done) -> IsoForest Outlier Removal -> Robust Scaling -> Binary Encoding (high-card) + One-Hot (low-card)**


In [ ]:
# Step 1: Remove outliers with IsoForest_0.05
df_final = df_imputed[~outlier_masks['IsoForest_0.05']].copy().reset_index(drop=True)
print(f"After outlier removal: {df_final.shape}")

# Step 2: Robust Scaling on numerical columns
scaler_final = RobustScaler()
df_final[numerical_cols] = scaler_final.fit_transform(df_final[numerical_cols])

# Step 3: Binary Encoding for all categorical columns (best VIF: mean=4.21, high VIF=1)
def binary_encode(df, col):
    unique_vals = sorted(df[col].dropna().unique())
    n_bits = max(1, int(np.ceil(np.log2(len(unique_vals) + 1))))
    mapping = {v: i + 1 for i, v in enumerate(unique_vals)}
    encoded = df[col].map(mapping).fillna(0).astype(int).to_numpy()
    return pd.DataFrame(
        {f"{col}_b{j}": (encoded >> j) & 1 for j in range(n_bits)},
        index=df.index
    )

encode_cols = [c for c in categorical_cols if c != target]
for col in encode_cols:
    bin_df = binary_encode(df_final, col)
    df_final = pd.concat([df_final.drop(columns=[col]), bin_df], axis=1)
print(f"After binary encoding: {df_final.shape}")

# Step 4: Move target to last column
cols = [c for c in df_final.columns if c != target] + [target]
df_final = df_final[cols]

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Class distribution:\n{df_final[target].value_counts()}")

df_final.to_csv('../data/processed/final_preprocessed.csv', index=False)
print("\nSaved: ../data/processed/final_preprocessed.csv")
